# Profiling 结果分析

04.02 从概念和代理实验两个层面分析了 RMSNorm 融合算子的收益来源：原生路径由 8 个通用 ACLNN 组成，每次完整尺寸前向需下发 8 次 launch、累计申请约 72 MiB + 16 KiB 的临时张量；融合路径则压缩为单次 aclnnRmsNorm launch，累计申请降至 8 MiB + 8 KiB，理论节省约 64 MiB + 8 KiB。通过标量 add 实验估算，8→1 launch 可节省约 100 μs 的调度开销。本节利用 Ascend Profiler 对 baseline 和 fused 训练任务的真实 profile 数据进行解析，验证这些理论预测是否与实际执行吻合。

## 运行分析脚本

以下 Python 单元读取两组目录中的 `trace_view.json`（完整时序图）、`api_statistic.csv` （aclnn api 调用） 和 `operator_details.csv`（算子执行记录），验证 RMSNorm 融合算子是否真的生效，以及 04.02 中提出的“8 次下发变 1 次”、“内存申请减少”等理论预测是否在真实硬件上成立。


In [ ]:
import os
original_dir = original_dir if 'original_dir' in globals() else os.getcwd()
%cd /mnt/workspace/torchtitan-npu
%env BASH_ENV=/home/developer/Ascend/cann/set_env.sh

In [ ]:
"""
Profiling结果分析脚本：验证RMSNorm融合算子收益（04.02理论 vs 04.04实测）

读取Ascend Profiler输出的三类数据：
  1. trace_view.json      - 完整时序事件，用于提取单次算子的内部下发链
  2. api_statistic.csv    - ACLNN API统计，观察Pow/Mul等碎片化调用是否减少
  3. operator_details.csv - NPU设备端耗时，对比融合前后kernel执行时间

分析策略分四层：
  - cpu_ranges: 扫描框架层RMSNorm相关事件（如npu::npu_rms_norm），确认converter生效
  - api_stats: 统计指定ACLNN API的总调用次数与耗时，验证碎片化通用op是否减少
  - device_ranges: 比较同层（同为225次调用）的device total duration，
       计算真实硬件加速比。
  - nested_rmsnorm_chain: 按时间窗口精准截取单次(2,1024,2048)调用的嵌套子事件，
       直接打印内部aclnn调用链（原生8个 vs 融合1个），证明"8→1 launch"理论
"""

import csv
import json
from collections import defaultdict
from pathlib import Path

BASELINE = Path(
    "./outputs/profile_traces/baseline"
)
FUSED = Path(
    "./outputs/profile_traces/fused_op"
)

API_NAMES = (
    "aclnnPowTensorScalar",
    "aclnnMul",
    "aclnnInplaceMul",
    "aclnnRmsNorm",
    "aclnnRmsNormGrad",
)
DETAIL_NAMES = ("aten::rms_norm", *API_NAMES)


def number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0


def find_trace(root):
    matches = list(root.glob("**/trace_view.json"))
    if len(matches) != 1:
        raise RuntimeError(
            f"{root}: expected one trace_view.json, found {len(matches)}"
        )
    return matches[0]


def read_csv(path):
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def summarize(root):
    trace = find_trace(root)
    data = json.loads(trace.read_text(encoding="utf-8"))
    events = data if isinstance(data, list) else data.get("traceEvents", [])

    cpu_ranges = defaultdict(lambda: {"count": 0, "time_us": 0.0})
    for event in events:
        name = str(event.get("name", ""))
        if event.get("cat") == "cpu_op" and (
            "rms_norm" in name.lower() or "rmsnorm" in name.lower()
        ):
            cpu_ranges[name]["count"] += 1
            cpu_ranges[name]["time_us"] += number(event.get("dur"))

    api_stats = defaultdict(lambda: {"count": 0, "time_us": 0.0})
    for row in read_csv(trace.parent / "api_statistic.csv"):
        name = row.get("API Name", "")
        if name in API_NAMES:
            api_stats[name]["count"] += int(number(row.get("Count")))
            api_stats[name]["time_us"] += number(row.get("Time(us)"))

    device_ranges = defaultdict(lambda: {"count": 0, "time_us": 0.0})
    for row in read_csv(trace.parent / "operator_details.csv"):
        name = row.get("Name", "")
        if name in DETAIL_NAMES:
            device_ranges[name]["count"] += 1
            device_ranges[name]["time_us"] += number(
                row.get("Device Total Duration(us)")
            )

    return {
        "cpu_ranges": cpu_ranges,
        "api_stats": api_stats,
        "device_ranges": device_ranges,
    }


def change(before, after):
    percent = "n/a" if before == 0 else f"{(after - before) / before * 100:+.1f}%"
    return f"{before:.2f} → {after:.2f} ({after - before:+.2f}, {percent})"


baseline = summarize(BASELINE)
fused = summarize(FUSED)

print("RMSNorm CPU ranges")
for label, report in (("BASELINE", baseline), ("FUSED", fused)):
    print(f"\n{label}")
    for name, row in report["cpu_ranges"].items():
        print(f"  {name}: {row['count']} calls, {row['time_us']:.1f} us")

print("\nRMSNorm-related APIs")
for name in API_NAMES:
    before = baseline["api_stats"][name]
    after = fused["api_stats"][name]
    print(
        f"  {name}: calls {change(before['count'], after['count'])}; "
        f"time {change(before['time_us'], after['time_us'])} us"
    )

print("\nDevice ranges")
for label, report in (("BASELINE", baseline), ("FUSED", fused)):
    for name, row in report["device_ranges"].items():
        average = row["time_us"] / row["count"]
        print(
            f"  {label} {name}: {row['count']} calls, "
            f"{row['time_us']:.1f} us total, {average:.1f} us avg"
        )


def nested_rmsnorm_chain(root, outer_name):
    trace = find_trace(root)
    data = json.loads(trace.read_text(encoding="utf-8"))
    events = data if isinstance(data, list) else data.get("traceEvents", [])
    outer = next(
        event
        for event in events
        if event.get("cat") == "cpu_op"
        and event.get("name") == outer_name
        and "2,1024,2048"
        in str(event.get("args", {}).get("Input Dims", "")).replace(" ", "")
    )
    start = float(outer["ts"])
    end = start + float(outer.get("dur", 0))
    nested = [
        event
        for event in events
        if event is not outer
        and event.get("pid") == outer.get("pid")
        and event.get("tid") == outer.get("tid")
        and float(event.get("ts", -1)) >= start
        and float(event.get("ts", -1)) + float(event.get("dur", 0) or 0) <= end
    ]
    nested.sort(key=lambda event: float(event.get("ts", 0)))
    launches = [
        event["name"]
        for event in nested
        if event.get("cat") == "cpu_op"
        and str(event.get("name", "")).startswith("aclnn")
    ]
    allocation_helpers = sum(
        event.get("name") in {"empty_tensor", "aten::empty", "aten::empty_strided"}
        for event in nested
    )
    return launches, allocation_helpers


baseline_chain, baseline_allocs = nested_rmsnorm_chain(BASELINE, "aten::rms_norm")
fused_chain, fused_allocs = nested_rmsnorm_chain(FUSED, "npu::npu_rms_norm")
print("\nOne (2,1024,2048) RMSNorm launch chain")
print(f"  BASELINE ({len(baseline_chain)}): {' -> '.join(baseline_chain)}")
print(f"  FUSED    ({len(fused_chain)}): {' -> '.join(fused_chain)}")
print(f"  eliminated ACLNN launches per forward: {len(baseline_chain) - len(fused_chain)}")
print(f"  allocation-helper ranges: {baseline_allocs} -> {fused_allocs}")


基于以上 profiling 数据分析，我们得出以下结论：

### 融合生效与机制验证

运行时 profiler 数据确认了融合转换的实际生效。与 baseline 中仅出现 `aten::rms_norm` 和 `aten::_fused_rms_norm` 不同，fused 版本明确捕获到 NPU 专用内核调用：前向的 `npu::npu_rms_norm` / `aclnnRmsNorm`，以及反向的 `npu::npu_rms_norm_backward` / `aclnnRmsNormGrad`。统计数据显示 `aclnnRmsNorm` 调用 225 次，`aclnnRmsNormGrad` 调用 113 次，证明 converter 的模块替换在运行时完整生效。

在指令级层面，通过对输入 shape 为 `(2,1024,2048)` 的单次 RMSNorm range 进行展开，验证了 8→1 的 launch 缩减：baseline 的 ACLNN 调用链为 `InplaceCopy → PowTensorScalar → Mean → InplaceAdds → Rsqrt → Mul → Mul → InplaceCopy`，共 8 次 launch；fused 仅剩单次 `aclnnRmsNorm`。这一结果直接证实了 04.02 的理论预测，即每次完整尺寸前向减少 7 次 ACLNN launch。

该机制缩减直接反映在设备端执行时间上。同为 225 次调用的前向 device range 中，baseline 的 `aten::rms_norm` 聚合耗时 26,504.9 μs，平均 117.8 μs/次；fused 的 `aclnnRmsNorm` 降至 8,380.3 μs，平均 37.2 μs/次。单次前向节省 80.6 μs，降幅达 68.4%。这一实测值与 04.02 基于 launch-only 模型的估算（~100 μs）处于同一量级；偏差可能源自 shape 覆盖范围、测量口径（设备执行时间 vs 完整同步耗时）以及实际带宽与理论峰值的差异。

### 端到端计算阶段收益

将视角放大到单个训练 step 的全局统计：

| 指标 | Baseline | Fused | 变化 |
|---|---|---:|---:|
| 单步总耗时 | 5,446.6 ms | 5,187.6 ms | **-4.8%** |
| Computing 阶段耗时 | 426.5 ms | 386.6 ms | **-9.4%** |
| TPS | ~387 | ~404 | **+4.4%** |
| 设备 op 总数 | 11,801 | 8,418 | -28.7% |
| 设备 op 总耗时 | 448.8 ms | 412.9 ms | -8.0% |
| CPU op event 总数 | 76,995 | 57,858 | -24.9% |

Computing 阶段 9.4% 的降幅是衡量此次优化成效的核心指标。它说明 RMSNorm 融合在计算子图内部产生了实质性影响，设备 op 总数下降 28.7%，设备 op 总耗时下降 8.0%，与 Computing 阶段降幅吻合，进一步印证了计算工作量的真实缩减。

### 收益归因：从调度瓶颈到计算密度提升

Baseline 中 `aten::rms_norm` 的 CPU 侧总耗时（146.6 ms）是其设备侧耗时（26.1 ms）的 5.6 倍，说明原生实现的性能瓶颈主要不在设备计算能力，而在于多次 launch 和中间张量分配带来的 CPU 调度压力。融合后，对于改算子，前向 CPU 耗时骤降至 2.4 ms，而设备耗时仍有 8.4 ms，调度开销已不再主导，耗时重心转移至设备侧执行。

碎片化 API 的变化为此提供了旁证（表内耗时为 CPU 侧 API 下发耗时）：

| API | 调用次数变化 | API 耗时变化 |
|---|---:|---:|
| `aclnnPowTensorScalar` | 762 → 311（-59.2%） | 6,629 → 2,478 μs（-62.6%） |
| `aclnnMul` | 1,576 → 448（-71.6%） | 16,156 → 6,419 μs（-60.3%） |
| `aclnnInplaceMul` | 310 → 310（0%） | 2,145 → 2,125 μs（-1.0%） |

Pow 和 Mul 的调用量及耗时显著下降，说明原本分散的逐元素平方、缩放等操作被融合内核吸收，计算密度得到提升，通用 API 的碎片化调用大幅减少。

### 总结

综上，04.04 的 profiling 实测完整印证了 04.02 的收益模型：RMSNorm 融合通过消除 8→1 launch 的 CPU 调度开销，并减少中间张量管理，在 Computing 阶段实现了 9.4% 的降幅，对于单算子替换而言，该优化成果显著。


## 练习

1. （判断题）确认融合算子生效时，应同时检查 fused trace 中目标 `npu::npu_rms_norm` 事件和 CSV 中的 device-op/API 统计。

2. （判断题）模块总数应从当前模型的 `named_modules()` 统计，不应从旧 profiling 表中的调用次数反推。

3. （单选题）本次真实输出中，Step Stage 的变化约为：
    A. -4.8%
    B. -28.7%
    C. +1.9%
    D. 0%


In [ ]:
%cd $original_dir
!cat ./answer/04.04_answer.txt